In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
q1 = "I just discovered the course. Can I still join it?"
q2 = "I just found out about the program. Can I still enroll?"

In [4]:
q3 = "How do I run Docker on Windows?"

In [5]:
v1 = model.encode(q1)
v2 = model.encode(q2)
v3 = model.encode(q3)

In [6]:
v1

array([-9.47482139e-03, -6.92323893e-02, -2.90595274e-02,  1.29389903e-02,
        1.36228632e-02,  2.34317587e-04, -7.74169564e-02, -9.13696885e-02,
       -1.04661331e-01, -1.92236584e-02, -4.30460349e-02,  3.96478735e-02,
        4.32519335e-03,  4.92471717e-02,  8.18583369e-03, -4.18449976e-02,
       -4.33822684e-02, -2.53526643e-02, -1.31611235e-03, -1.77624042e-03,
       -8.88451114e-02,  4.49002311e-02, -2.61511914e-02,  2.34496072e-02,
       -9.18066129e-03,  1.37693435e-02,  1.85691845e-02,  8.78783166e-02,
       -3.21309045e-02, -7.98438638e-02, -3.69027667e-02,  6.97171018e-02,
        3.12004853e-02,  2.90625524e-02,  4.98373574e-03,  1.73434224e-02,
        6.40965104e-02, -5.67701198e-02,  6.59304950e-03,  2.26624236e-02,
       -4.27380353e-02, -2.78819669e-02, -1.23384660e-02,  5.00044636e-02,
        3.09627876e-02,  9.94023755e-02, -5.98819330e-02, -8.57653096e-02,
        1.95483807e-02,  3.08334157e-02,  2.59876698e-02, -4.66156118e-02,
       -3.99187353e-04,  

In [7]:
len(v1), len(v2), len(v3)

(384, 384, 384)

In [8]:
v1.dot(v2) # cosine similarity

np.float32(0.62813616)

In [9]:
v1.dot(v3)

np.float32(-0.076364584)

In [10]:
v2.dot(v1)

np.float32(0.62813616)

In [11]:
from ingest import load_faq_data
documents = load_faq_data()

In [12]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [13]:
len(documents)

1350

In [14]:
# building text from documnents
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [16]:
len(texts)

1350

In [17]:
from tqdm.auto import tqdm

In [18]:
1350/50

27.0

In [23]:
# We won't hand the model all of them at once. That takes a long time, 
# and we can't see what's happening inside. Instead we split them into batches.

In [19]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [22]:
len(vectors[0])

384

In [24]:
import numpy as np
X = np.array(vectors)

In [29]:
X.size

518400

In [26]:
len(X)

1350

In [27]:
X

array([[-0.02670619, -0.12245757,  0.01594414, ..., -0.00230649,
        -0.11218392, -0.02365565],
       [-0.01099562, -0.11074749, -0.02536941, ...,  0.09022235,
        -0.02697364,  0.01975661],
       [-0.08896554, -0.06128183,  0.00775607, ...,  0.04059711,
         0.00479284, -0.02745946],
       ...,
       [-0.03652921,  0.01415428, -0.06838642, ...,  0.04316788,
         0.08105536, -0.02148629],
       [-0.13091598, -0.06990604, -0.00931881, ..., -0.00044338,
        -0.01285727,  0.01426919],
       [-0.07984785,  0.01926977,  0.02544985, ..., -0.0336803 ,
        -0.01884027,  0.05837052]], shape=(1350, 384), dtype=float32)

In [30]:
X.shape

(1350, 384)

#### Vector Search

In [33]:
q2

'I just found out about the program. Can I still enroll?'

In [34]:
# v2 is the vector representation of q2
scores = X.dot(v2)
# same as 
# scores = [v_query.dot(X[i]) for i in range(len(X))]

In [35]:
len(scores)

1350

In [36]:
scores

array([ 0.2913355 ,  0.1483143 ,  0.45367575, ...,  0.02022233,
        0.10223855, -0.0518817 ], shape=(1350,), dtype=float32)

In [38]:
# Best match
idx = np.argmax(scores)
idx, scores[idx], documents[idx]

(np.int64(538),
 np.float32(0.5665001),
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'})

In [40]:
# for q1
print(q1)
scores = X.dot(v1)
idx = np.argmax(scores)
print(scores[idx])
print(idx)
documents[idx]

I just discovered the course. Can I still join it?
0.8365049
538


{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [41]:
# for q3
print(q3)
scores = X.dot(v3)
idx = np.argmax(scores)
print(scores[idx])
print(idx)
documents[idx]

How do I run Docker on Windows?
0.65365976
65


{'course': 'data-engineering-zoomcamp',
 'section': 'Module 1: Docker',
 'question': 'Docker: The input device is not a TTY (Docker run for Windows)',
 'answer': 'You may encounter this error:\n\n```bash\n$ docker run -it ubuntu bash\nthe input device is not a TTY. If you are using mintty, try prefixing the command with \'winpty\'\n```\n\nSolution:\n\n- Use `winpty` before the Docker command:\n  \n  ```bash\n  $ winpty docker run -it ubuntu bash\n  ```\n\n- Alternatively, create an alias:\n  \n  ```bash\n  echo "alias docker=\'winpty docker\'" >> ~/.bashrc\n  ```\n  \n  or\n  \n  ```bash\n  echo "alias docker=\'winpty docker\'" >> ~/.bash_profile\n  ```\n  \nSource: [Stack Overflow](https://stackoverflow.com/a/49965690)',
 'doc_id': 'ed8dcfbb5a'}

In [46]:
np.argsort(scores)[-5:][::-1]

array([  65,  696, 1081,   60,   64])

In [49]:
## Top 5 results
top5 = np.argsort(scores)[-5:][::-1]

In [52]:
for idx in top5:
    print(scores[idx])
    print(documents[idx]['question'], documents[idx]['answer'])
    print()

0.65365976
Docker: The input device is not a TTY (Docker run for Windows) You may encounter this error:

```bash
$ docker run -it ubuntu bash
the input device is not a TTY. If you are using mintty, try prefixing the command with 'winpty'
```

Solution:

- Use `winpty` before the Docker command:
  
  ```bash
  $ winpty docker run -it ubuntu bash
  ```

- Alternatively, create an alias:
  
  ```bash
  echo "alias docker='winpty docker'" >> ~/.bashrc
  ```
  
  or
  
  ```bash
  echo "alias docker='winpty docker'" >> ~/.bash_profile
  ```
  
Source: [Stack Overflow](https://stackoverflow.com/a/49965690)

0.62733185
Install docker in WSL2 without installing Docker Desktop If you want to install Docker in WSL2 on Windows without Docker Desktop, follow these steps:

1. **Install Docker**

   You can ignore the warnings during installation.
   
   ```bash
   curl -fsSL https://get.docker.com -o get-docker.sh
   sudo sh get-docker.sh
   ```
   
2. **Add Your User to the Docker Group**
   
   `